In [2]:
# Run once: install pyyaml if not already installed
!pip install pyyaml

In [3]:

#Finding where am I
import os
os.environ['USE_LOCALSTACK'] = '1'
os.environ['LOCALSTACK_URL'] = 'http://192.168.1.30:4566'
os.chdir('/config/workspace/python/pythonawsproject')
print(os.getcwd())

# Now import (after chdir so Python can find the network package)
from network.state import load_state
print(load_state())

/config/workspace/python/pythonawsproject
{'vpc_id': 'vpc-086f174b', 'vpc_name': 'dev-vpc', 'public-subnet-az1': 'subnet-bb5cad5d', 'public-subnet-az2': 'subnet-92febc26', 'private-subnet-az1': 'subnet-d3dcfd50', 'private-subnet-az2': 'subnet-5cb4937f', 'igw_id': 'igw-ac703f85', 'public_rt_id': 'rtb-f4da44dc', 'public-subnet-az1_rt_assoc': 'rtbassoc-9a37ac81', 'public-subnet-az2_rt_assoc': 'rtbassoc-b8b898c5', 'private_rt_id': 'rtb-9425dc59', 'private-subnet-az1_rt_assoc': 'rtbassoc-c996fd19', 'private-subnet-az2_rt_assoc': 'rtbassoc-334a6637', 'nat_eip_alloc_id': 'eipalloc-f15f5bb0c19a51730', 'nat_id': 'nat-5418ea87da75c0603', 'private_nat_route': True, 'sg_dev-web-sg': 'sg-5c85da35302d46462'}


In [4]:
#What's the plan yo!
import yaml

with open('config/network.yaml', 'r') as f:
    config = yaml.safe_load(f) #//putting all the values from load state to config

print(config['vpc'])
print(config['security_groups'])

{'name': 'dev-vpc', 'cidr': '10.0.0.0/16', 'region': 'ap-south-1'}
[{'name': 'dev-web-sg', 'description': 'Allow SSH, HTTP, HTTPS inbound', 'ingress': [{'protocol': 'tcp', 'port': 22, 'cidr': '0.0.0.0/0', 'description': 'SSH access'}, {'protocol': 'tcp', 'port': 80, 'cidr': '0.0.0.0/0', 'description': 'HTTP'}, {'protocol': 'tcp', 'port': 443, 'cidr': '0.0.0.0/0', 'description': 'HTTPs'}], 'egress': [{'protocol': 'tcp', 'port': 80, 'cidr': '0.0.0.0/0', 'description': 'HTTP'}]}]


In [5]:
##Now it can work
from network.vpc import create_vpc

vpc_id = create_vpc(config)
print(f"Result: {vpc_id}")

Created VPC: vpc-086f174b
VPC is now avilable
Tagged VPC with: ['Name', 'ManagedBy', 'Layer', 'Environment']
State saved
Result: vpc-086f174b


In [5]:
from network.subnets import create_subnet
##We are utting full config python file here for execution
result = create_subnet(config)
print(result)

[{'name': 'public-subnet-az1', 'cidr': '10.0.1.0/24', 'az': 'ap-south-1a'}, {'name': 'public-subnet-az2', 'cidr': '10.0.2.0/24', 'az': 'ap-south-1b'}, {'name': 'private-subnet-az1', 'cidr': '10.0.3.0/24', 'az': 'ap-south-1a'}, {'name': 'private-subnet-az2', 'cidr': '10.0.4.0/24', 'az': 'ap-south-1b'}]
Subnet already Exsists by name: public-subnet-az1
Subnet already Exsists by name: public-subnet-az2
Subnet already Exsists by name: private-subnet-az1
Subnet already Exsists by name: private-subnet-az2
All subnets done.
{'vpc_id': 'vpc-086f174b', 'vpc_name': 'dev-vpc', 'public-subnet-az1': 'subnet-bb5cad5d', 'public-subnet-az2': 'subnet-92febc26', 'private-subnet-az1': 'subnet-d3dcfd50', 'private-subnet-az2': 'subnet-5cb4937f', 'igw_id': 'igw-ac703f85', 'public_rt_id': 'rtb-f4da44dc', 'public-subnet-az1_rt_assoc': 'rtbassoc-9a37ac81', 'public-subnet-az2_rt_assoc': 'rtbassoc-b8b898c5', 'private_rt_id': 'rtb-9425dc59', 'private-subnet-az1_rt_assoc': 'rtbassoc-c996fd19', 'private-subnet-az2_

In [8]:
from network.internet_gateway  import creating_igw
from network.state import load_state, save_state
from network.aws_client import get_client

igw_id = creating_igw(config)
print(f"Result: {igw_id}")
# ec2 = get_client('ec2')
# response = ec2.describe_internet_gateways(
#     Filters=[{'Name': 'attachment.vpc-id', 'Values': [vpc_id]}]
# )

# print(vpc_id)

# response = ec2.describe_internet_gateways()
# print(response['InternetGateways'])

# igw_id = response['InternetGateways'][0]['InternetGatewayId']
# print(f"Found exsisting IGW: {igw_id}")

# ec2.attach_internet_gateway(InternetGatewayId=igw_id, VpcId=vpc_id)
# print(f"Attached {igw_id} to {vpc_id}")

# ##Saving and loading of state file
# state = load_state()
# state['igw_id'] = igw_id
# save_state(state)
# print("Saved State")

# igw_id = creating_igw(config)
# print(f"Result: {igw_id}")


IGW already exsists: igw-ac703f85
Result: igw-ac703f85


In [9]:
from network.route_tables import create_route_tables

result = create_route_tables(config)

Public route table already exsists: rtb-f4da44dc
 Already associated: public-subnet-az1
 Already associated: public-subnet-az2
Private route table already exists: rtb-9425dc59
  Already associated: private-subnet-az1
  Already associated: private-subnet-az2
Route tables done.


In [10]:
from network.nat_gateway import create_nat_gateway

result = create_nat_gateway(config)

Elastic IP already exists: eipalloc-f15f5bb0c19a51730
NAT Gateway already exists: nat-5418ea87da75c0603
Private route to NAT already exists
NAT Gateway done.


In [11]:
from network.state import load_state, save_state

state = load_state()
del state['nat_id']
del state['nat_eip_alloc_id']

In [12]:
import os
os.chdir('/config/workspace/python/pythonawsproject')
print(os.listdir('network/'))


['internet_gateway.py', 'state.py', 'security_groups.py', 'planner.py', 'route_tables.py', 'main.py', 'nat_gateway.py', '__init__.py', 'ec2_instance.py', 'subnets.py', 'aws_client.py', '__pycache__', 'vpc.py']


In [13]:
from network.security_groups import create_security_groups
create_security_groups(config)

Security Group already exists: sg-5c85da35302d46462 (dev-web-sg)
Security Groups done.


{'vpc_id': 'vpc-086f174b',
 'vpc_name': 'dev-vpc',
 'public-subnet-az1': 'subnet-bb5cad5d',
 'public-subnet-az2': 'subnet-92febc26',
 'private-subnet-az1': 'subnet-d3dcfd50',
 'private-subnet-az2': 'subnet-5cb4937f',
 'igw_id': 'igw-ac703f85',
 'public_rt_id': 'rtb-f4da44dc',
 'public-subnet-az1_rt_assoc': 'rtbassoc-9a37ac81',
 'public-subnet-az2_rt_assoc': 'rtbassoc-b8b898c5',
 'private_rt_id': 'rtb-9425dc59',
 'private-subnet-az1_rt_assoc': 'rtbassoc-c996fd19',
 'private-subnet-az2_rt_assoc': 'rtbassoc-334a6637',
 'nat_eip_alloc_id': 'eipalloc-f15f5bb0c19a51730',
 'nat_id': 'nat-5418ea87da75c0603',
 'private_nat_route': True,
 'sg_dev-web-sg': 'sg-5c85da35302d46462'}

In [16]:
from network.ec2_instance import create_ec2_instance
instance_id = create_ec2_instance(config)
print(f"Result: {instance_id}")

Launched EC2 instance: i-c073d8e05fe2fbfff
Waiting for instance to reach 'running' state...
Instance is now running.
Save EC@ instance to state: i-c073d8e05fe2fbfff
Result: i-c073d8e05fe2fbfff


In [17]:
from network.planner import run_planner
plan = run_planner()

🔍 Verifying resources against AWS...

  ✅ No drift — state matches AWS.

  EXECUTION PLAN
  ✅ vpc                  "dev-vpc" → EXISTS (vpc-086f174b...)
  ✅ subnet               "public-subnet-az1" → EXISTS (subnet-bb5cad5d...)
  ✅ subnet               "public-subnet-az2" → EXISTS (subnet-92febc26...)
  ✅ subnet               "private-subnet-az1" → EXISTS (subnet-d3dcfd50...)
  ✅ subnet               "private-subnet-az2" → EXISTS (subnet-5cb4937f...)
  ✅ internet_gateway     "dev-igw" → EXISTS (igw-ac703f85...)
  ✅ nat_gateway          "dev-nat" → EXISTS (nat-5418ea87da75c060...)
  ✅ route_table          "public-route-table" → EXISTS (rtb-f4da44dc...)
  ✅ route_table          "private-route-table" → EXISTS (rtb-9425dc59...)
  ✅ security_group       "dev-web-sg" → EXISTS (sg-5c85da35302d46462...)
  ✅ ec2_instance         "dev-web-server" → EXISTS (i-c073d8e05fe2fbfff...)
  0 to create, 0 to delete, 11 unchanged

  ✅ Infrastructure matches desired state. Nothing to do.


In [18]:
from network.main import command_plan
command_plan(config)


 Scanning the  battlefiend that is config file
🔍 Verifying resources against AWS...

  ✅ No drift — state matches AWS.

  EXECUTION PLAN
  ✅ vpc                  "dev-vpc" → EXISTS (vpc-086f174b...)
  ✅ subnet               "public-subnet-az1" → EXISTS (subnet-bb5cad5d...)
  ✅ subnet               "public-subnet-az2" → EXISTS (subnet-92febc26...)
  ✅ subnet               "private-subnet-az1" → EXISTS (subnet-d3dcfd50...)
  ✅ subnet               "private-subnet-az2" → EXISTS (subnet-5cb4937f...)
  ✅ internet_gateway     "dev-igw" → EXISTS (igw-ac703f85...)
  ✅ nat_gateway          "dev-nat" → EXISTS (nat-5418ea87da75c060...)
  ✅ route_table          "public-route-table" → EXISTS (rtb-f4da44dc...)
  ✅ route_table          "private-route-table" → EXISTS (rtb-9425dc59...)
  ✅ security_group       "dev-web-sg" → EXISTS (sg-5c85da35302d46462...)
  ✅ ec2_instance         "dev-web-server" → EXISTS (i-c073d8e05fe2fbfff...)
  0 to create, 0 to delete, 11 unchanged

  ✅ Infrastructure matches d

[{'type': 'vpc', 'name': 'dev-vpc', 'action': 'NO-OP', 'id': 'vpc-086f174b'},
 {'type': 'subnet',
  'name': 'public-subnet-az1',
  'action': 'NO-OP',
  'id': 'subnet-bb5cad5d'},
 {'type': 'subnet',
  'name': 'public-subnet-az2',
  'action': 'NO-OP',
  'id': 'subnet-92febc26'},
 {'type': 'subnet',
  'name': 'private-subnet-az1',
  'action': 'NO-OP',
  'id': 'subnet-d3dcfd50'},
 {'type': 'subnet',
  'name': 'private-subnet-az2',
  'action': 'NO-OP',
  'id': 'subnet-5cb4937f'},
 {'type': 'internet_gateway',
  'name': 'dev-igw',
  'action': 'NO-OP',
  'id': 'igw-ac703f85'},
 {'type': 'nat_gateway',
  'name': 'dev-nat',
  'action': 'NO-OP',
  'id': 'nat-5418ea87da75c0603'},
 {'type': 'route_table',
  'name': 'public-route-table',
  'action': 'NO-OP',
  'id': 'rtb-f4da44dc'},
 {'type': 'route_table',
  'name': 'private-route-table',
  'action': 'NO-OP',
  'id': 'rtb-9425dc59'},
 {'type': 'security_group',
  'name': 'dev-web-sg',
  'action': 'NO-OP',
  'id': 'sg-5c85da35302d46462'},
 {'type':

In [20]:
from network.main import command_destroy
command_destroy(config)


 RETREAT ORDER - Demolishing infrastructure ...

====== Terminate EC2 Instance ======
. Waiting for i-c073d8e05fe2fbfff to terminate...
===== Destroying NAT Gateway =====
  Waiting for NAT nat-5418ea87da75c0603 to die...
  NAT Gateway deleted.
===== Releasing Elastic IP =====
  EIP released. Thanks for choosing the service.
===== Disassociating Route Tables =====
  Unfollowing: public-subnet-az1_rt_assoc
  Unfollowing: public-subnet-az2_rt_assoc
  Unfollowing: private-subnet-az1_rt_assoc
  Unfollowing: private-subnet-az2_rt_assoc
===== Deleting Route Tables =====
  Shredding: rtb-f4da44dc
  Shredding: rtb-9425dc59
===== Detaching + Deleting IGW =====
  IGW igw-ac703f85 unhooked from VPC.
  IGW igw-ac703f85 deleted.
  IGW gone. No more front door.
 Deleteing Security Group: sg-5c85da35302d46462 (sg_dev-web-sg)
===== Deleting Subnets =====
  Demolishing: subnet-bb5cad5d (public-subnet-az1)
  Demolishing: subnet-92febc26 (public-subnet-az2)
  Demolishing: subnet-d3dcfd50 (private-subnet-

In [ ]:
command_plan(config)